# Vérification / renumérotation des actes à partir des zones (JJ167+)

Pour les registres à partir de JJ167, le tableau d'actes ne donne plus systématiquement
le folio de début de chaque acte. Ce notebook reconstitue une numérotation de contrôle
à partir des zones **AI** et **AC** (dans l'ordre de lecture : folio, puis position verticale),
la confronte aux repères connus du tableau d'actes, et produit un rapport HTML interactif
permettant de pointer/corriger la numérotation au fil du travail.

Ce notebook est autonome : il recharge et reconstruit `df_images`, `df_actes`, `df_zones`
à partir des fichiers sources (mêmes fonctions que le notebook principal
`JJ100-139-versImportLabelStudio2.ipynb`), puis applique les fonctions de vérification.


# Paramètres

In [281]:
import os
import re

corpus = 'JJ200-JJ211'   # à adapter au corpus traité
# JJ160-JJ169
# JJ170-JJ179
# JJ180-JJ199
# JJ200-JJ211



# Chemins des fichiers sources
INPUT_ZONES  = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio.csv"       # fichier 1 : zones YOLO
INPUT_ACTES  = "../List-of-acts/Acts-JJ96-JJ195_enriched_Locus-20260521.xlsm"                  # fichier 2 : liste des actes
INPUT_IMAGES = f"../List-of-images/{corpus}_image_data.csv"    # fichier 3 : images téléchargées

# Dossier de sortie
OUT_DIR = f"{corpus}/output"
os.makedirs(OUT_DIR, exist_ok=True)

# Labels de zones attendus par le modèle d'acte
LABELS_ATTENDUS = {'AC', 'AI', 'AM', 'AF', 'NIA', 'Table'}


def parse_corpus(corpus):
    match = re.match(r'([A-Z]+)(\d+)-([A-Z]+)(\d+)', corpus)
    if not match:
        raise ValueError(f"Format inattendu : {corpus}")
    prefix = match.group(1)
    start, end = int(match.group(2)), int(match.group(4))
    return {f"{prefix}{i}" for i in range(start, end + 1)}

REGISTRES_CIBLES = parse_corpus(corpus)
print(REGISTRES_CIBLES)

# Séparateur CSV des fichiers sources ('\t' = tabulation)
SEP = '\t'


{'JJ205', 'JJ210', 'JJ211', 'JJ201', 'JJ200', 'JJ209', 'JJ208', 'JJ207', 'JJ203', 'JJ204', 'JJ202', 'JJ206'}


# Imports et fonctions utilitaires (identiques au notebook principal)

In [282]:
import os
import re
import ast
import json
import warnings
import pandas as pd
import numpy as np
from collections import defaultdict


pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)


def normalize_path(p):
    """Normalise les séparateurs Windows/Unix et renvoie le basename."""
    return os.path.basename(str(p).replace("\\\\", "/"))


def extract_register(folder_path):
    """
    Extrait le numéro de registre normalisé depuis un chemin.
    'Paris_Archives_Nationales_JJ096' → 'JJ96' (sans zéro initial).
    Cible le DERNIER composant du chemin contenant JJ + chiffres.
    """
    parts = str(folder_path).replace("\\\\", "/").split("/")
    for part in reversed(parts):
        m = re.search(r'JJ0*(\d+)$', part)
        if m:
            return f"JJ{m.group(1)}"
    return None


def normalize_folio(raw):
    """
    Normalise un label de folio pour jointure.
      '6'        -> '6r'
      '12v'      -> '12v'
      '1r'       -> '1r'
      '103bis'   -> '103bisr'
      '103bis v' -> '103bisv'
      '103 bis recto' -> '103bisr'
      '103 bis verso' -> '103bisv'
      'plat supérieur' -> 'plat supérieur'
    """
    s = str(raw).strip().lower()

    if s in ('vacat', 'vacatr', 'vacatv'):
        return None   # sera logué comme "sans folio" -> ignoré proprement

    if not s or s == 'nan':
        return None

    # Normaliser les variantes textuelles de recto/verso
    s = s.replace('recto', 'r').replace('verso', 'v')

    # Supprimer les espaces autour de 'bis' et avant r/v
    s = re.sub(r'\s+bis\s*', 'bis', s)
    s = re.sub(r'\s+([rv])$', r'\1', s)

    # Cas standard : chiffres + optionnel 'bis' + r ou v
    if re.match(r'^\d+(?:bis)?[rv]$', s):
        return s

    # Chiffres + optionnel 'bis' sans suffixe -> recto par défaut
    if re.match(r'^\d+(?:bis)?$', s):
        return s + 'r'

    return s


def parse_abs_coords(coord_str):
    """Parse '1,23,3866,6279' -> [x, y, w, h] ou None si invalide."""
    try:
        parts = [int(v) for v in str(coord_str).split(',')]
        if len(parts) == 4:
            return parts
    except Exception:
        pass
    return None

def parse_image_stem(image_path):
    stem = normalize_path(image_path).rsplit('.', 1)[0]  # retire .jpg
    parts = stem.split('_')
    volume = parts[-2]               # 'JJ096'
    folio_sort_key = int(parts[-1])  # 100
    return volume, folio_sort_key


def safe_to_int(series):
    return (
        series
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .round()
        .astype(int)
    )


def strip_leading_zeros(folio_norm):
    """'001r' -> '1r', '012bisv' -> '12bisv' (règle spécifique JJ126, réutilisable ailleurs)."""
    if not folio_norm:
        return folio_norm
    m = re.match(r'^0*(\d+)(bis)?([rv])$', folio_norm)
    if m:
        return f"{m.group(1)}{m.group(2) or ''}{m.group(3)}"
    return folio_norm


# Chargement des fichiers CSV sources

## Table `images.csv`

In [283]:
skipped = {}

with warnings.catch_warnings(record=True) as w_images:
    warnings.simplefilter("always")
    df_images_raw = pd.read_csv(INPUT_IMAGES, sep=',', dtype=str,
                             low_memory=False, on_bad_lines='warn', encoding='utf-8')
    skipped['images'] = len(w_images)

print(f"Images chargées : {len(df_images_raw):>6} lignes  ({skipped['images']} ligne(s) ignorée(s))")

df_images = df_images_raw[[
    'manifestURL', 'canvasId', 'urlImage', 'imageLabel',
    'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded',
    'urlResizedImage', 'ResizedImageWidthAsDownloaded',
    'ResizedImageHeightAsDownloaded', 'folderPath'
]].copy()

df_images['image_id']       = range(1, len(df_images) + 1)
df_images['image_filename'] = df_images['imageFileName'].apply(normalize_path)
df_images[['volume', 'folio_sort_key']] = df_images['image_filename'].apply(
    lambda p: pd.Series(parse_image_stem(p))
    )
df_images['register']       = df_images['folderPath'].apply(extract_register)
df_images['folio_norm']     = df_images['imageLabel'].apply(normalize_folio)

df_images.loc[df_images['volume'] == 'JJ181', 'folio_norm'] = (
    df_images.loc[df_images['volume'] == 'JJ181', 'folio_norm'].apply(strip_leading_zeros))


df_images.rename(columns={
    'manifestURL':                    'manifest_url',
    'canvasId':                       'canvas_id',
    'urlImage':                       'url_full',
    'imageLabel':                     'folio_label',
    'imageWidthAsDownloaded':         'width_px',
    'imageHeightAsDownloaded':        'height_px',
    'urlResizedImage':                'url_resized',
    'ResizedImageWidthAsDownloaded':  'resized_width_px',
    'ResizedImageHeightAsDownloaded': 'resized_height_px',
}, inplace=True)

df_images.drop(columns=['imageFileName', 'folderPath'], inplace=True)

IMAGES_COLS = [
    'image_id', 'register', 'image_filename', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'width_px', 'height_px', 'url_full', 'url_resized',
    'resized_width_px', 'resized_height_px', 'manifest_url', 'canvas_id',
]
df_images = df_images[IMAGES_COLS]

df_images.to_csv(os.path.join(OUT_DIR, "images.csv"), index=False, sep=SEP)
print(f"images.csv -> {len(df_images)} lignes, {len(df_images.columns)} colonnes")

# Index (volume, folio_sort_key) -> [image_id, ...] (ordre séquentiel préservé)
img_by_folio = defaultdict(list)
for _, row in df_images.iterrows():
    if pd.notna(row['volume']) and pd.notna(row['folio_sort_key']):
        img_by_folio[(row['volume'], int(row['folio_sort_key']))].append(row['image_id'])

images_by_register = defaultdict(list)
for _, row in df_images.iterrows():
    if row['register']:
        images_by_register[row['register']].append(row['image_id'])

print(f"Index img_by_folio    : {len(img_by_folio)} clés (volume, folio_sort_key)")
print(f"Registres distincts   : {list(images_by_register.keys())}")

df_images.head(3)


Images chargées :   3950 lignes  (0 ligne(s) ignorée(s))
images.csv -> 3950 lignes, 15 colonnes
Index img_by_folio    : 3950 clés (volume, folio_sort_key)
Registres distincts   : ['JJ200', 'JJ201', 'JJ202', 'JJ203', 'JJ204', 'JJ205', 'JJ206', 'JJ207', 'JJ208', 'JJ209', 'JJ210', 'JJ211']


,image_id,register,image_filename,volume,folio_sort_key,folio_label,folio_norm,width_px,height_px,url_full,url_resized,resized_width_px,resized_height_px,manifest_url,canvas_id
0,1,JJ200,Paris_Archives_Nationales_JJ200_1.jpg,JJ200,1,garde supérieure,garde supérieure,4257,4215,https://iiif.irht.cnrs.fr/iiif/ark:/63955/v7xa53s0mymd/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/v7xa53s0mymd/full/,1200/0/default.jpg",1212,1200,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290330
1,2,JJ200,Paris_Archives_Nationales_JJ200_2.jpg,JJ200,2,contre garde supérieure,contre garde supérieure,4257,4209,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqq1lebcoo3i/full/,1200/0/default.jpg",1214,1200,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290331
2,3,JJ200,Paris_Archives_Nationales_JJ200_3.jpg,JJ200,3,1r,1r,3801,4503,https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/full/0/default.jpg,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg",1200,1422,https://api.irht.cnrs.fr/ark:/63955/frpbm744s3r5/manifest.json,https://arca.irht.cnrs.fr/iiif/128501/canvas/canvas-4290332


## Table `actes.csv`

In [284]:
skipped = {}

with warnings.catch_warnings(record=True) as w_actes:
    warnings.simplefilter("always")
    df_actes_raw = pd.read_excel(INPUT_ACTES)
    skipped['actes'] = len(w_actes)

print(f"Actes  chargés  : {len(df_actes_raw):>6} lignes  ({skipped['actes']} ligne(s) ignorée(s))")

df_actes = df_actes_raw.copy()

df_actes.rename(columns={
    'ID-temporaire':        'acte_id',
    'Register':             'volume',
    'Act_number':           'act_number',
    'Nvelle numérotation':  'new_numbering',
    'Folio Number ou page': 'folio_raw',
    'vérif':                'verified',
    'Note':                 'note',
}, inplace=True)

df_actes = df_actes[df_actes['volume'].isin(REGISTRES_CIBLES)].copy()
print(f"Actes après filtre {corpus} : {len(df_actes)} lignes")
print(df_actes['volume'].value_counts().sort_index())

df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)
df_actes.loc[df_actes['volume'] == 'JJ181', 'folio_norm'] = (
    df_actes.loc[df_actes['volume'] == 'JJ181', 'folio_norm'].apply(strip_leading_zeros))

# Table de correspondance folio_norm -> folio_sort_key par registre
folio_to_sortkey = (df_images[['volume', 'folio_norm', 'folio_sort_key']]
                    .drop_duplicates(subset=['volume', 'folio_norm'])
                    .reset_index(drop=True))

df_actes = df_actes.merge(
    folio_to_sortkey,
    on=['volume', 'folio_norm'],
    how='left'
)

unmatched = df_actes[df_actes['folio_sort_key'].isna()]
if not unmatched.empty:
    print(f"AVERTISSEMENT : {len(unmatched)} acte(s) sans folio_sort_key :")
    print(unmatched[['acte_id', 'volume', 'folio_norm']].to_string(index=False))
else:
    print("Tous les folios ont été matchés.")

# Aplatir les concordances : chaque inventaire source donne 3 colonnes
concordance_cols = [c for c in df_actes.columns if '.xml_' in c]
inventory_sources = defaultdict(list)
for col in concordance_cols:
    src = col.split('.xml_')[0] + '.xml'
    inventory_sources[src].append(col)

for src, cols in inventory_sources.items():
    short = re.sub(r'^Paris_AN_JJ_inventaire_', '', src.replace('.xml', ''))
    short = short.replace('Guerin_tome1-tome12', 'Guerin')
    for suffix, key in [('_Act_number', 'act'), ('_Head', 'head'), ('_Locus', 'locus')]:
        col = next((c for c in cols if c.endswith(suffix)), None)
        if col:
            df_actes[f'conc_{short}_{key}'] = df_actes[col].fillna('')

BASE_COLS = ['acte_id', 'volume', 'act_number', 'new_numbering',
             'folio_raw', 'folio_norm', 'verified', 'note']
CONC_COLS = [c for c in df_actes.columns if c.startswith('conc_')]
df_actes_out = df_actes[BASE_COLS + CONC_COLS].copy()

df_actes_out.to_csv(os.path.join(OUT_DIR, "actes.csv"), index=False, sep=SEP)
print(f"actes.csv -> {len(df_actes_out)} lignes, {len(df_actes_out.columns)} colonnes")
df_actes_out.head(8)


Actes  chargés  :  46530 lignes  (0 ligne(s) ignorée(s))
Actes après filtre JJ200-JJ211 : 4454 lignes
volume
JJ200     215
JJ201     216
JJ202     111
JJ203      81
JJ204     189
JJ205     500
JJ206    1184
JJ207     379
JJ208     258
JJ209     303
JJ210     266
JJ211     752
Name: count, dtype: int64
AVERTISSEMENT : 4454 acte(s) sans folio_sort_key :
 acte_id volume folio_norm
 42592.0  JJ200       None
 42593.0  JJ200       None
 42594.0  JJ200       None
 42595.0  JJ200       None
 42596.0  JJ200       None
 42597.0  JJ200       None
 42598.0  JJ200       None
 42599.0  JJ200       None
 42600.0  JJ200       None
 42601.0  JJ200       None
 42602.0  JJ200       None
 42603.0  JJ200       None
 42604.0  JJ200       None
 42605.0  JJ200       None
 42606.0  JJ200       None
 42607.0  JJ200       None
 42608.0  JJ200       None
 42609.0  JJ200       None
 42610.0  JJ200       None
 42611.0  JJ200       None
 42612.0  JJ200       None
 42613.0  JJ200       None
 42614.0  JJ200       Non

,acte_id,volume,act_number,new_numbering,folio_raw,folio_norm,verified,note,conc_Guerin_act,conc_Guerin_head,conc_Guerin_locus,conc_Longnon_act,conc_Longnon_head,conc_Longnon_locus,conc_Viard_act,conc_Viard_head,conc_Viard_locus,conc_Gascogne_act,conc_Gascogne_head,conc_Gascogne_locus,conc_Languedoc_act,conc_Languedoc_head,conc_Languedoc_locus,conc_Loire_act,conc_Loire_head,conc_Loire_locus,conc_Rouergue_act,conc_Rouergue_head,conc_Rouergue_locus,conc_IR421-JJA-JJ79A_act,conc_IR421-JJA-JJ79A_head,conc_IR421-JJA-JJ79A_locus,conc_IR422-JJ80-155_act,conc_IR422-JJ80-155_head,conc_IR422-JJ80-155_locus,conc_IR423-JJ156-211_act,conc_IR423-JJ156-211_head,conc_IR423-JJ156-211_locus
0,42592.0,JJ200,1,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,42593.0,JJ200,2,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,42594.0,JJ200,3,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,42595.0,JJ200,4,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,42596.0,JJ200,5,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
5,42597.0,JJ200,6,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
6,42598.0,JJ200,7,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
7,42599.0,JJ200,8,NaN,NaN,None,NaN,NaN,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


## Table `zones.csv`

In [285]:
skipped = {}

with warnings.catch_warnings(record=True) as w_zones:
    warnings.simplefilter("always")
    df_zones_raw = pd.read_csv(INPUT_ZONES, sep=',', dtype=str,
                               low_memory=False, on_bad_lines='warn')
    skipped['zones'] = len(w_zones)

# Normalisation des noms de colonnes -> le reste du code utilise toujours
# 'volume', 'folio_sort_key', peu importe le corpus
df_zones_raw = df_zones_raw.rename(columns={
    'registre': 'volume',
    'ordre':    'folio_sort_key',
})

def parse_labelstudio_rects(label_str):
    if pd.isna(label_str):
        return []
    if not str(label_str).strip():
        return []
    try:
        data = json.loads(label_str)
    except Exception:
        try:
            data = ast.literal_eval(label_str)
        except Exception:
            return []
    if not isinstance(data, list):
        return []

    rows = []
    for r in data:
        if 'rectanglelabels' not in r:
            continue
        rows.append({
            'class_name': r['rectanglelabels'][0],
            'x_pct': r['x'],
            'y_pct': r['y'],
            'w_pct': r['width'],
            'h_pct': r['height'],
            'image_width_px': r['original_width'],
            'image_height_px': r['original_height'],
        })
    return rows

# Supprime lignes entièrement vides ou sans annotation
df_zones_raw = df_zones_raw.dropna(how='all')
df_zones_raw = df_zones_raw.dropna(subset=['label'])
df_zones_raw = df_zones_raw[
    df_zones_raw['label'].astype(str).str.strip() != ''
]
print(f"Zones importées : {len(df_zones_raw)}")

df_zones_raw['url_image_full'] = df_zones_raw['image']

df_zones_raw['volume'] = df_zones_raw['volume'].apply(
    lambda r: re.sub(r'JJ(\d+)', lambda m: f"JJ{int(m.group(1)):03d}", str(r))
)
df_zones_raw['folio_sort_key'] = pd.to_numeric(df_zones_raw['folio_sort_key'], errors='coerce')

rows = []
for _, r in df_zones_raw.iterrows():
    annotations = parse_labelstudio_rects(r['label'])
    for ann in annotations:
        rows.append({
            'url_image_full':   r['url_image_full'],
            'image_path':       r['image_path'],
            'image_filename':   r['image'],
            'volume':           r['volume'],
            'folio_sort_key':   r['folio_sort_key'],
            'class_name':       ann['class_name'],
            'x_pct':            ann['x_pct'],
            'y_pct':            ann['y_pct'],
            'w_pct':            ann['w_pct'],
            'h_pct':            ann['h_pct'],
            'image_width_px':   ann['image_width_px'],
            'image_height_px':  ann['image_height_px'],
        })

df_zones = pd.DataFrame(rows)

df_zones['abs_x'] = safe_to_int(df_zones['x_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_y'] = safe_to_int(df_zones['y_pct'] / 100.0 * df_zones['image_height_px'])
df_zones['abs_w'] = safe_to_int(df_zones['w_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_h'] = safe_to_int(df_zones['h_pct'] / 100.0 * df_zones['image_height_px'])

df_zones = df_zones.sort_values(
    by=['volume', 'folio_sort_key', 'abs_y', 'abs_x'],
    ascending=[True, True, True, True]
    )

img_lookup = df_images[['image_id', 'volume', 'folio_sort_key',
                         'folio_label', 'folio_norm']].copy()

df_zones = df_zones.merge(
    img_lookup,
    on=['volume', 'folio_sort_key'],
    how='left'
)

df_zones['zone_id'] = np.arange(1, len(df_zones) + 1)

CLASS_MAP = {
    'AC': 0,
    'AI': 1,
    'AF': 2,
    'AM': 3,
    'NIA': 4,
    'Table': 5,
}

df_zones['class_id'] = df_zones['class_name'].map(CLASS_MAP)

ZONES_COLS = [
    'zone_id', 'image_id', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'class_id', 'class_name',
    'abs_x', 'abs_y', 'abs_w', 'abs_h',
    'x_pct', 'y_pct', 'w_pct', 'h_pct',
    'url_image_full', 'image_filename',
    'image_width_px', 'image_height_px',
]
df_zones = df_zones[ZONES_COLS]

df_zones.to_csv(os.path.join(OUT_DIR, "zones.csv"), index=False, sep=SEP)
print(f"zones.csv -> {len(df_zones)} lignes, {len(df_zones.columns)} colonnes")
df_zones.head(10)


Zones importées : 3875
zones.csv -> 7161 lignes, 20 colonnes


,zone_id,image_id,volume,folio_sort_key,folio_label,folio_norm,class_id,class_name,abs_x,abs_y,abs_w,abs_h,x_pct,y_pct,w_pct,h_pct,url_image_full,image_filename,image_width_px,image_height_px
0,1,3,JJ200,3,1r,1r,1,AI,6,3,1194,1419,0.485,0.180,99.510,99.820000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vlnsr0equbys/full/1200,/0/default.jpg",1200,1422
1,2,4,JJ200,4,1v,1v,2,AF,1,0,1197,502,0.100,0.010,99.720,37.080000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg",1200,1355
2,3,4,JJ200,4,1v,1v,1,AI,0,502,1198,846,0.000,37.080,99.860,62.460000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vm3ve3icqw2h/full/1200,/0/default.jpg",1200,1355
3,4,5,JJ200,5,2r,2r,3,AM,2,1,1198,1458,0.170,0.085,99.800,99.870000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxnv80kx42jr/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxnv80kx42jr/full/1200,/0/default.jpg",1200,1460
4,5,6,JJ200,6,2v,2v,2,AF,2,1,1198,1092,0.155,0.070,99.845,79.750525,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg",1200,1369
5,6,6,JJ200,6,2v,2v,1,AI,0,1113,1200,254,0.000,81.335,100.000,18.570000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vvbr91mlze72/full/1200,/0/default.jpg",1200,1369
6,7,7,JJ200,7,3r,3r,2,AF,0,1,1198,713,0.005,0.040,99.810,48.920000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg",1200,1458
7,8,7,JJ200,7,3r,3r,1,AI,0,713,1198,744,0.000,48.935,99.820,51.030000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vam6syxo1fmp/full/1200,/0/default.jpg",1200,1458
8,9,8,JJ200,8,3v,3v,2,AF,3,0,1196,1091,0.215,0.005,99.650,79.470000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbvqvsuxbmsm/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbvqvsuxbmsm/full/1200,/0/default.jpg",1200,1373
9,10,8,JJ200,8,3v,3v,1,AI,1,1078,1197,295,0.095,78.510,99.770,21.480000,"https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbvqvsuxbmsm/full/1200,/0/default.jpg","https://iiif.irht.cnrs.fr/iiif/ark:/63955/vbvqvsuxbmsm/full/1200,/0/default.jpg",1200,1373


# Fonctions de vérification / renumérotation

## Construction de l'URL IIIF d'une région

In [286]:
def build_iiif_region_url(url_full, x_pct, y_pct, w_pct, h_pct):
    """
    Construit l'URL IIIF d'une région à partir de l'URL de l'image pleine,
    en utilisant directement les pourcentages Label Studio (région IIIF 'pct:').
    On évite ainsi tout recalcul en pixels absolus, qui suppose à tort que
    'original_width'/'original_height' (résolution de l'image annotée dans
    Label Studio) correspond à la résolution de l'image IIIF plein format --
    ce qui n'est pas le cas (ex. image servie en /full/1200,/ alors que le
    plein format est plus grand).
    Structure IIIF : {id}/{region}/{size}/{rotation}/{quality}.{format}
    On remplace uniquement le paramètre 'region' (toujours 'full' en entrée),
    identifié par sa position -- les 3 segments suivants (size/rotation/quality.format)
    sont préservés tels quels.
    """
    region = f"pct:{x_pct},{y_pct},{w_pct},{h_pct}"
    return re.sub(
        r'/full/([^/]+/[^/]+/[^/]+)$',
        f'/{region}/\\1',
        str(url_full)
    )


## Numérotation calculée depuis les zones AI/AC

In [287]:
def build_zone_sequence(df_zones, volume):
    """
    Séquence des actes calculée depuis les zones AI/AC d'un volume,
    dans l'ordre de lecture (folio, puis position verticale/horizontale sur la page).
    Une zone AI = un acte qui commence sur ce folio (suite éventuelle sur les folios
    suivants via AM/AF, déjà gérée ailleurs dans le pipeline de liaison).
    Une zone AC = un acte complet tenant sur le folio.
    """
    g = (df_zones[(df_zones['volume'] == volume) &
                  (df_zones['class_name'].isin(['AI', 'AC']))]
         .sort_values(['folio_sort_key', 'abs_y', 'abs_x'])
         .reset_index(drop=True)
         .copy())

    g['seq_num']       = range(1, len(g) + 1)
    g['iiif_url_page'] = g['url_image_full']   # image entière (contexte du folio)
    g['iiif_url_zone'] = g.apply(
        lambda r: build_iiif_region_url(r['url_image_full'], r['x_pct'], r['y_pct'],
                                         r['w_pct'], r['h_pct']), axis=1)

    return g[['seq_num', 'zone_id', 'image_id', 'folio_sort_key', 'folio_norm',
              'class_name', 'iiif_url_page', 'iiif_url_zone']]


## Confrontation à trois numérotations — plusieurs actes par folio

**Le tableau d'actes comporte une numérotation "Act Number" déjà existante, mais
irrégulière** (numéros "bis", sauts, parfois sans folio renseigné à partir de
JJ167). Trois colonnes sont donc mises en regard pour chaque zone AI/AC détectée :

1. **`#`** : rang de la zone AI/AC dans l'ordre de lecture du volume, de 1 à N
   (c'est la numérotation "calculée" pure, indépendante du tableau).
2. **`Act Number` (positionnel)** : valeur brute de la colonne *Act Number* du
   tableau, prise dans l'ordre des lignes du fichier Excel et appariée
   directement, rang par rang, à la zone détectée — avec ses "bis" ou
   numéros irréguliers conservés tels quels. Si le tableau comporte moins de
   lignes que de zones détectées, les rangs excédentaires restent vides.
3. **`ActNumber/Folio`** : vérification indépendante par ancrage sur le folio,
   quand celui-ci est renseigné dans le tableau (repère fiable mais
   sporadique). Cette colonne confirme — ou contredit — l'hypothèse
   d'appariement positionnel de la colonne 2.

Une divergence entre les colonnes 2 et 3 à un même rang signale un probable
décalage de numérotation (zone manquante ou en trop avant ce rang), et
déclenche un avertissement. Comme précédemment, le cas de plusieurs actes
(1 AI + un ou plusieurs AC) sur un même folio est géré : les repères d'un
même folio sont appariés dans l'ordre où ils apparaissent dans le tableau
source.

In [288]:
def cross_check_sequence(df_seq, df_actes, volume):
    """
    Trois numérotations sont mises en regard pour un volume :

      1. df_seq['seq_num'] (colonne '#' à l'affichage) : rang de la zone AI/AC
         dans l'ordre de lecture, de 1 à N.
      2. act_number_positional : valeur brute de 'Act Number' prise dans
         l'ordre des lignes du fichier Excel, appariée rang par rang à la
         zone détectée (numéros bis/irréguliers conservés tels quels). Vide
         au-delà du nombre de lignes disponibles dans le tableau.
      3. act_number_anchor : vérification indépendante par ancrage sur le
         folio, quand celui-ci est renseigné (repère fiable mais sporadique).

    Une divergence entre (2) et (3) à un même rang signale un décalage de
    numérotation probable (zone manquante ou en trop avant ce rang).
    """
    df_seq = df_seq.copy().reset_index(drop=True)
    df_seq['act_number_positional'] = pd.NA
    df_seq['act_number_anchor']     = pd.NA
    df_seq['warning']               = ''

    actes_vol = (df_actes[df_actes['volume'] == volume]
                 .reset_index(drop=True)
                 .copy())
    actes_vol['excel_order'] = actes_vol.index  # ordre d'apparition dans le fichier source

    # --- 1) Appariement positionnel simple, rang par rang ---
    n = min(len(df_seq), len(actes_vol))
    for i in range(n):
        df_seq.loc[i, 'act_number_positional'] = actes_vol.loc[i, 'act_number']
    # au-delà de len(actes_vol) : la colonne reste vide (pd.NA), comme demandé

    # --- 2) Ancrage indépendant par folio (repère fiable mais sporadique) ---
    reperes = actes_vol[actes_vol['folio_norm'].notna() & actes_vol['act_number'].notna()].copy()

    for folio_norm, actes_folio in reperes.groupby('folio_norm'):
        actes_folio = actes_folio.sort_values('excel_order')
        zones_folio_idx = df_seq.index[df_seq['folio_norm'] == folio_norm].tolist()

        if not zones_folio_idx:
            continue

        n_actes = len(actes_folio)
        n_zones = len(zones_folio_idx)

        if n_actes != n_zones:
            msg = (f"AMBIGU sur folio {folio_norm} : {n_actes} acte(s) au tableau "
                   f"vs {n_zones} zone(s) AI/AC détectée(s)")
            df_seq.loc[zones_folio_idx[0], 'warning'] = msg

        for act_number, idx in zip(actes_folio['act_number'].tolist(), zones_folio_idx):
            df_seq.loc[idx, 'act_number_anchor'] = act_number

    # --- 3) Divergence entre appariement positionnel et ancrage vérifié ---
    for idx, r in df_seq.iterrows():
        if pd.notna(r['act_number_anchor']) and pd.notna(r['act_number_positional']):
            if str(r['act_number_anchor']) != str(r['act_number_positional']):
                msg = (f"decalage : Act Number positionnel '{r['act_number_positional']}' "
                       f"!= ancrage folio '{r['act_number_anchor']}'")
                existing = df_seq.loc[idx, 'warning']
                df_seq.loc[idx, 'warning'] = (existing + " | " + msg) if existing else msg

    return df_seq


## Export HTML interactif

In [289]:
import json


def export_interactive_html(df_seq, volume, output_path):
    """
    Exporte un rapport HTML pour vérifier la numérotation des actes d'un volume,
    à partir du DataFrame produit par cross_check_sequence().
    """
    records = df_seq.copy()
    records = records.where(records.notna(), None)
    data = records.to_dict('records')

    html = HTML_TEMPLATE.replace('__VOLUME__', volume).replace(
        '__DATA__', json.dumps(data, ensure_ascii=False))

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"Rapport généré : {output_path}")


HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Vérification des actes — __VOLUME__</title>
<style>
  body { font-family: -apple-system, Segoe UI, Arial, sans-serif; margin: 20px; background:#fafafa; }
  h1 { font-size: 1.3em; }
  .meta { color:#555; margin-bottom: 14px; }
  table { border-collapse: collapse; width: 100%; font-size: 13px; background:#fff; }
  th, td { border: 1px solid #ddd; padding: 5px 8px; text-align: left; vertical-align: middle; }
  th { background: #2c3e50; color: #fff; position: sticky; top: 0; }
  tr.mismatch { background: #f8d7da; }
  tr.warning { background: #fff3cd; }
  tr.anchor { background: #e8f4ea; }
  td.thumb img { height: 40px; border:1px solid #999; cursor: pointer; }
  td.thumb.page img { height: 60px; }
  button { cursor: pointer; }
  .toolbar { margin-bottom: 10px; }
  .warn-text { color:#a15c00; font-weight:600; }
  .lightbox { display:none; position:fixed; inset:0; background:rgba(0,0,0,.85);
              align-items:center; justify-content:center; z-index:10; }
  .lightbox img { max-height:90vh; max-width:90vw; }
</style>
</head>
<body>

<h1>Vérification des actes — __VOLUME__</h1>
<div class="meta">
  <b>#</b> : rang de la zone AI/AC dans l'ordre de lecture (1 à N).
  <b>Act Number</b> : valeur du tableau appariée positionnellement, rang par rang
  (numéros "bis"/irréguliers conservés ; vide si le tableau n'a plus de ligne).
  <b>ActNumber/Folio</b> : vérification indépendante par ancrage sur le folio,
  quand il est renseigné.
  <br>Une ligne en <span style="background:#f8d7da; padding:0 4px;">rouge</span> signale une
  divergence entre ces deux colonnes (décalage de numérotation probable).
</div>

<div class="toolbar">
  <button onclick="exportCsv()">Exporter CSV</button>
  <span id="summary" style="margin-left:16px; font-weight:600;"></span>
</div>

<table id="tbl">
  <thead>
    <tr>
      <th>#</th>
      <th>Act Number</th>
      <th>ActNumber/Folio</th>
      <th>Folio</th>
      <th>Type</th>
      <th>Folio (contexte)</th>
      <th>Zone</th>
      <th>Avertissement</th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>

<div class="lightbox" id="lightbox" onclick="this.style.display='none'">
  <img id="lightbox-img" src="">
</div>

<script>
let rows = __DATA__;

function render() {
  const tbody = document.getElementById('tbody');
  tbody.innerHTML = '';
  let warnCount = 0;

  rows.forEach((r) => {
    const tr = document.createElement('tr');
    if (r.warning && r.warning.includes('decalage')) {
      tr.classList.add('mismatch');
    } else if (r.warning) {
      tr.classList.add('warning');
    }
    if (r.act_number_anchor !== null && r.act_number_anchor !== undefined) tr.classList.add('anchor');
    if (r.warning) warnCount += 1;

    tr.innerHTML = `
      <td>${r.seq_num ?? ''}</td>
      <td>${r.act_number_positional ?? ''}</td>
      <td>${r.act_number_anchor ?? ''}</td>
      <td>${r.folio_norm ?? ''}</td>
      <td>${r.class_name ?? ''}</td>
      <td class="thumb page">${r.iiif_url_page ? `<img src="${r.iiif_url_page}" onclick="showLightbox('${r.iiif_url_page}')">` : ''}</td>
      <td class="thumb">${r.iiif_url_zone ? `<img src="${r.iiif_url_zone}" onclick="showLightbox('${r.iiif_url_zone}')">` : ''}</td>
      <td class="warn-text">${r.warning ?? ''}</td>
    `;
    tbody.appendChild(tr);
  });

  document.getElementById('summary').textContent =
    `${rows.length} zone(s) AI/AC — ${warnCount} avertissement(s)`;
}

function showLightbox(url) {
  document.getElementById('lightbox-img').src = url;
  document.getElementById('lightbox').style.display = 'flex';
}

function exportCsv() {
  const header = ['seq_num','act_number_positional','act_number_anchor','folio_norm','class_name',
                   'iiif_url_page','iiif_url_zone','warning'];
  const lines = [header.join(';')];
  rows.forEach(r => {
    lines.push([
      r.seq_num ?? '', r.act_number_positional ?? '', r.act_number_anchor ?? '', r.folio_norm ?? '',
      r.class_name ?? '', r.iiif_url_page ?? '', r.iiif_url_zone ?? '',
      (r.warning ?? '').replace(/;/g,',')
    ].join(';'));
  });
  const blob = new Blob([lines.join('\\n')], {type: 'text/csv;charset=utf-8;'});
  const a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = 'verification_actes___VOLUME__.csv';
  a.click();
}

render();
</script>
</body>
</html>
"""


# Exécution — tous les volumes du corpus

In [290]:
rapports = {}
for vol in sorted(REGISTRES_CIBLES):
    seq = build_zone_sequence(df_zones, vol)
    if seq.empty:
        continue
    seq = cross_check_sequence(seq, df_actes_out, vol)
    rapports[vol] = seq
    export_interactive_html(seq, vol, os.path.join(OUT_DIR, f'verification_actes_{vol}.html'))

resume = pd.DataFrame([
    {
        'volume': vol,
        'nb_actes_calcules': len(seq),
        'nb_reperes_folio_connus': seq['act_number_anchor'].notna().sum(),
        'nb_avertissements': (seq['warning'] != '').sum(),
    }
    for vol, seq in rapports.items()
])
resume


Rapport généré : JJ200-JJ211/output\verification_actes_JJ200.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ201.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ202.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ203.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ204.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ205.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ206.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ207.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ208.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ209.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ210.html
Rapport généré : JJ200-JJ211/output\verification_actes_JJ211.html


,volume,nb_actes_calcules,nb_reperes_folio_connus,nb_avertissements
0,JJ200,215,0,0
1,JJ201,216,0,0
2,JJ202,110,0,0
3,JJ203,81,0,0
4,JJ204,189,0,0
5,JJ205,500,0,0
6,JJ206,1184,0,0
7,JJ207,379,0,0
8,JJ208,258,0,0
9,JJ209,303,0,0
